In [0]:
from pyspark.sql import functions as F

CATALOG = "healthcare_analytics"
GOLD = "gold"


def gold_table(name):
    return spark.table(f"{CATALOG}.{GOLD}.{name}")


def save_mart(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{GOLD}.{table_name}")
    )
    print(f"Created {CATALOG}.{GOLD}.{table_name}")

In [0]:
from pyspark.sql import functions as F

CATALOG = "healthcare_analytics"
GOLD = "gold"


def gold_table(name):
    return spark.table(f"{CATALOG}.{GOLD}.{name}")


def save_mart(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{CATALOG}.{GOLD}.{table_name}")
    )
    print(f"Created {CATALOG}.{GOLD}.{table_name}")

In [0]:
enc = gold_table("fact_encounter")
facility = gold_table("dim_facility")

operations = (
    enc
    .join(
        facility.select(
            "facility_key",
            "organization_name",
            "city",
            "state"
        ),
        "facility_key",
        "left"
    )
    .withColumn(
        "month_start",
        F.trunc(F.to_date("encounter_start"), "month")
    )
    .groupBy(
        "facility_key",
        "organization_name",
        "city",
        "state",
        "month_start"
    )
    .agg(
        F.count("*").alias("encounter_count"),
        F.countDistinct("patient_id").alias("unique_patients"),

        F.sum(
            F.when(
                F.lower(F.col("encounter_class")) == "inpatient",
                1
            ).otherwise(0)
        ).alias("inpatient_encounters"),

        F.sum(
            F.when(
                F.lower(F.col("encounter_class")) == "emergency",
                1
            ).otherwise(0)
        ).alias("emergency_encounters"),

        F.sum(
            F.when(
                F.lower(F.col("encounter_class")) == "ambulatory",
                1
            ).otherwise(0)
        ).alias("ambulatory_encounters"),

        F.avg(
            "encounter_duration_hours"
        ).alias("avg_encounter_duration_hours"),

        F.sum(
            "total_claim_cost"
        ).alias("total_claim_cost"),

        F.avg(
            "total_claim_cost"
        ).alias("avg_cost_per_encounter"),

        F.sum(
            "payer_coverage"
        ).alias("payer_coverage"),

        F.sum(
            "patient_responsibility"
        ).alias("patient_responsibility")
    )
    .withColumn(
        "inpatient_pct",
        F.when(
            F.col("encounter_count") > 0,
            F.round(
                F.col("inpatient_encounters")
                / F.col("encounter_count") * 100,
                2
            )
        )
    )
)

save_mart(
    operations,
    "mart_hospital_operations"
)

Created healthcare_analytics.gold.mart_hospital_operations


In [0]:
readm = gold_table("fact_readmission")
facility = gold_table("dim_facility")

readmission_mart = (
    readm
    .join(
        facility.select(
            "facility_key",
            "organization_name",
            "city",
            "state"
        ),
        "facility_key",
        "left"
    )
    .withColumn(
        "month_start",
        F.trunc(
            F.to_date("encounter_end"),
            "month"
        )
    )
    .groupBy(
        "facility_key",
        "organization_name",
        "city",
        "state",
        "month_start"
    )
    .agg(
        F.count("*").alias(
            "eligible_inpatient_discharges"
        ),

        F.sum(
            "readmission_30d_flag"
        ).alias(
            "readmissions_30d"
        ),

        F.avg(
            F.when(
                F.col("readmission_30d_flag") == 1,
                F.col("days_to_readmission")
            )
        ).alias(
            "avg_days_to_readmission"
        ),

        F.sum(
            "total_claim_cost"
        ).alias(
            "index_encounter_cost"
        )
    )
    .withColumn(
        "readmission_rate_pct",
        F.round(
            F.when(
                F.col("eligible_inpatient_discharges") > 0,
                F.col("readmissions_30d")
                / F.col("eligible_inpatient_discharges")
                * 100
            ).otherwise(0),
            2
        )
    )
)

save_mart(
    readmission_mart,
    "mart_readmissions"
)

Created healthcare_analytics.gold.mart_readmissions


In [0]:
patients = gold_table("dim_patient")
enc = gold_table("fact_encounter")
readm = gold_table("fact_readmission")
conditions = spark.table(
    "healthcare_analytics.silver.conditions"
)

enc_patient = (
    enc
    .groupBy("patient_id")
    .agg(
        F.count("*").alias("encounter_count"),

        F.sum(
            F.when(
                F.col("is_inpatient") == True,
                1
            ).otherwise(0)
        ).alias("inpatient_encounter_count"),

        F.sum(
            F.when(
                F.lower(F.col("encounter_class")) == "emergency",
                1
            ).otherwise(0)
        ).alias("emergency_encounter_count"),

        F.sum(
            "total_claim_cost"
        ).alias("total_claim_cost"),

        F.sum(
            "payer_coverage"
        ).alias("payer_coverage"),

        F.sum(
            "patient_responsibility"
        ).alias("patient_responsibility")
    )
)

condition_patient = (
    conditions
    .groupBy("patient_id")
    .agg(
        F.countDistinct("code").alias(
            "distinct_condition_count"
        ),
        F.sum(
            F.when(
                F.col("is_active_condition") == True,
                1
            ).otherwise(0)
        ).alias(
            "active_condition_count"
        )
    )
)

readm_patient = (
    readm
    .groupBy("patient_id")
    .agg(
        F.sum(
            "readmission_30d_flag"
        ).alias("readmission_30d_count")
    )
)

outcomes = (
    patients
    .join(
        enc_patient,
        "patient_id",
        "left"
    )
    .join(
        condition_patient,
        "patient_id",
        "left"
    )
    .join(
        readm_patient,
        "patient_id",
        "left"
    )
    .fillna(
        0,
        subset=[
            "encounter_count",
            "inpatient_encounter_count",
            "emergency_encounter_count",
            "total_claim_cost",
            "payer_coverage",
            "patient_responsibility",
            "distinct_condition_count",
            "active_condition_count",
            "readmission_30d_count"
        ]
    )
    .withColumn(
        "had_30d_readmission",
        F.col("readmission_30d_count") > 0
    )
    .withColumn(
        "operational_risk_tier",

        F.when(
            (F.col("readmission_30d_count") >= 2)
            | (F.col("inpatient_encounter_count") >= 5),
            "High"
        )

        .when(
            (F.col("readmission_30d_count") >= 1)
            | (F.col("inpatient_encounter_count") >= 2)
            | (F.col("active_condition_count") >= 5),
            "Medium"
        )

        .otherwise("Standard")
    )
)

save_mart(
    outcomes,
    "mart_patient_outcomes"
)

Created healthcare_analytics.gold.mart_patient_outcomes


In [0]:
conditions = spark.table(
    "healthcare_analytics.silver.conditions"
)

procedures = spark.table(
    "healthcare_analytics.silver.procedures"
)

enc = gold_table("fact_encounter").select(
    "encounter_id",
    "encounter_start"
)

diagnosis_events = (
    conditions
    .join(
        enc,
        "encounter_id",
        "left"
    )
    .withColumn(
        "month_start",
        F.trunc(
            F.to_date("encounter_start"),
            "month"
        )
    )
    .groupBy(
        "month_start",
        "code",
        "description"
    )
    .agg(
        F.count("*").alias("event_count"),
        F.countDistinct(
            "patient_id"
        ).alias("unique_patients"),
        F.countDistinct(
            "encounter_id"
        ).alias("unique_encounters")
    )
    .withColumn(
        "clinical_event_type",
        F.lit("Diagnosis")
    )
    .withColumn(
        "total_event_cost",
        F.lit(0.0)
    )
)

procedure_events = (
    procedures
    .withColumn(
        "month_start",
        F.trunc(
            F.to_date("procedure_start"),
            "month"
        )
    )
    .groupBy(
        "month_start",
        "code",
        "description"
    )
    .agg(
        F.count("*").alias("event_count"),
        F.countDistinct(
            "patient_id"
        ).alias("unique_patients"),
        F.countDistinct(
            "encounter_id"
        ).alias("unique_encounters"),
        F.sum(
            "base_cost"
        ).alias("total_event_cost")
    )
    .withColumn(
        "clinical_event_type",
        F.lit("Procedure")
    )
)

clinical = (
    diagnosis_events
    .unionByName(
        procedure_events,
        allowMissingColumns=True
    )
)

save_mart(
    clinical,
    "mart_clinical_utilization"
)

Created healthcare_analytics.gold.mart_clinical_utilization


In [0]:
enc = gold_table("fact_encounter")
payer = gold_table("dim_payer")
facility = gold_table("dim_facility")

cost_mart = (
    enc
    .join(
        payer.select(
            "payer_key",
            "payer_name"
        ),
        "payer_key",
        "left"
    )
    .join(
        facility.select(
            "facility_key",
            "organization_name",
            "state"
        ),
        "facility_key",
        "left"
    )
    .withColumn(
        "month_start",
        F.trunc(
            F.to_date("encounter_start"),
            "month"
        )
    )
    .groupBy(
        "month_start",
        "payer_key",
        "payer_name",
        "facility_key",
        "organization_name",
        "state"
    )
    .agg(
        F.count("*").alias("claim_encounter_count"),

        F.countDistinct(
            "patient_id"
        ).alias("unique_patients"),

        F.sum(
            "total_claim_cost"
        ).alias("total_claim_cost"),

        F.avg(
            "total_claim_cost"
        ).alias("avg_claim_cost"),

        F.sum(
            "payer_coverage"
        ).alias("payer_coverage"),

        F.sum(
            "patient_responsibility"
        ).alias("patient_responsibility")
    )
    .withColumn(
        "payer_coverage_pct",
        F.round(
            F.when(
                F.col("total_claim_cost") > 0,
                F.col("payer_coverage")
                / F.col("total_claim_cost")
                * 100
            ).otherwise(0),
            2
        )
    )
)

save_mart(
    cost_mart,
    "mart_claims_cost"
)

Created healthcare_analytics.gold.mart_claims_cost


In [0]:
enc = gold_table("fact_encounter")
payer = gold_table("dim_payer")
facility = gold_table("dim_facility")

cost_mart = (
    enc
    .join(
        payer.select(
            "payer_key",
            "payer_name"
        ),
        "payer_key",
        "left"
    )
    .join(
        facility.select(
            "facility_key",
            "organization_name",
            "state"
        ),
        "facility_key",
        "left"
    )
    .withColumn(
        "month_start",
        F.trunc(
            F.to_date("encounter_start"),
            "month"
        )
    )
    .groupBy(
        "month_start",
        "payer_key",
        "payer_name",
        "facility_key",
        "organization_name",
        "state"
    )
    .agg(
        F.count("*").alias("claim_encounter_count"),

        F.countDistinct(
            "patient_id"
        ).alias("unique_patients"),

        F.sum(
            "total_claim_cost"
        ).alias("total_claim_cost"),

        F.avg(
            "total_claim_cost"
        ).alias("avg_claim_cost"),

        F.sum(
            "payer_coverage"
        ).alias("payer_coverage"),

        F.sum(
            "patient_responsibility"
        ).alias("patient_responsibility")
    )
    .withColumn(
        "payer_coverage_pct",
        F.round(
            F.when(
                F.col("total_claim_cost") > 0,
                F.col("payer_coverage")
                / F.col("total_claim_cost")
                * 100
            ).otherwise(0),
            2
        )
    )
)

save_mart(
    cost_mart,
    "mart_claims_cost"
)

Created healthcare_analytics.gold.mart_claims_cost


In [0]:
# ============================================================
# PROVIDER PERFORMANCE MART
# ============================================================

enc = gold_table("fact_encounter")
providers = gold_table("dim_provider")
facility = gold_table("dim_facility")
readm = gold_table("fact_readmission")


# ------------------------------------------------------------
# Encounter performance by provider + facility + month
# ------------------------------------------------------------

provider_enc = (
    enc
    .join(
        providers.select(
            "provider_key",
            "provider_name"
        ),
        "provider_key",
        "left"
    )
    .join(
        facility.select(
            "facility_key",
            "organization_name"
        ),
        "facility_key",
        "left"
    )
    .withColumn(
        "month_start",
        F.trunc(
            F.to_date("encounter_start"),
            "month"
        )
    )
    .groupBy(
        "provider_key",
        "provider_name",
        "facility_key",
        "organization_name",
        "month_start"
    )
    .agg(
        F.count("*").alias(
            "encounter_count"
        ),

        F.countDistinct(
            "patient_id"
        ).alias(
            "unique_patients"
        ),

        F.sum(
            F.when(
                F.col("is_inpatient") == True,
                1
            ).otherwise(0)
        ).alias(
            "inpatient_encounters"
        ),

        F.sum(
            F.when(
                F.lower(F.col("encounter_class")) == "emergency",
                1
            ).otherwise(0)
        ).alias(
            "emergency_encounters"
        ),

        F.avg(
            "encounter_duration_hours"
        ).alias(
            "avg_encounter_duration_hours"
        ),

        F.sum(
            "total_claim_cost"
        ).alias(
            "total_claim_cost"
        ),

        F.avg(
            "total_claim_cost"
        ).alias(
            "avg_cost_per_encounter"
        )
    )
)


# ------------------------------------------------------------
# Readmission performance by provider + facility + month
# ------------------------------------------------------------

provider_readm = (
    readm
    .withColumn(
        "month_start",
        F.trunc(
            F.to_date("encounter_start"),
            "month"
        )
    )
    .groupBy(
        "provider_key",
        "facility_key",
        "month_start"
    )
    .agg(
        F.count("*").alias(
            "eligible_inpatient_discharges"
        ),

        F.sum(
            "readmission_30d_flag"
        ).alias(
            "readmissions_30d"
        )
    )
)


# ------------------------------------------------------------
# Final Provider Performance Mart
# ------------------------------------------------------------

provider_performance = (
    provider_enc
    .join(
        provider_readm,
        [
            "provider_key",
            "facility_key",
            "month_start"
        ],
        "left"
    )
    .fillna(
        0,
        subset=[
            "eligible_inpatient_discharges",
            "readmissions_30d"
        ]
    )
    .withColumn(
        "readmission_rate_pct",
        F.round(
            F.when(
                F.col("eligible_inpatient_discharges") > 0,

                F.col("readmissions_30d")
                / F.col("eligible_inpatient_discharges")
                * 100

            ).otherwise(0),
            2
        )
    )
    .withColumn(
        "encounters_per_patient",
        F.round(
            F.when(
                F.col("unique_patients") > 0,

                F.col("encounter_count")
                / F.col("unique_patients")

            ).otherwise(0),
            2
        )
    )
)


save_mart(
    provider_performance,
    "mart_provider_performance"
)

display(
    provider_performance
    .orderBy(
        F.desc("encounter_count")
    )
)

Created healthcare_analytics.gold.mart_provider_performance


provider_key,facility_key,month_start,provider_name,organization_name,encounter_count,unique_patients,inpatient_encounters,emergency_encounters,avg_encounter_duration_hours,total_claim_cost,avg_cost_per_encounter,eligible_inpatient_discharges,readmissions_30d,readmission_rate_pct,encounters_per_patient
1dd4caf0181c01c6604d5651fa3a9b35b74d9b3c8247cf290ccb944de4afaeb4,116c4d5fd6fe6d897e7008c7497d6220e2ddc3f0c5f41dd34129b82f33ddfc61,2019-01-01,Ted955 Reilly981,Fitchburg Outpatient Clinic,51,5,1,1,4.048169934640523,35436.11,694.8256862745098,1,0,0.0,10.2
1dd4caf0181c01c6604d5651fa3a9b35b74d9b3c8247cf290ccb944de4afaeb4,116c4d5fd6fe6d897e7008c7497d6220e2ddc3f0c5f41dd34129b82f33ddfc61,2019-02-01,Ted955 Reilly981,Fitchburg Outpatient Clinic,43,3,1,1,1.3811046511627907,21619.70000000001,502.78372093023285,1,0,0.0,14.33
1dd4caf0181c01c6604d5651fa3a9b35b74d9b3c8247cf290ccb944de4afaeb4,116c4d5fd6fe6d897e7008c7497d6220e2ddc3f0c5f41dd34129b82f33ddfc61,2019-03-01,Ted955 Reilly981,Fitchburg Outpatient Clinic,43,3,1,1,1.9670478036175711,16698.829999999998,388.34488372093017,1,0,0.0,14.33
1dd4caf0181c01c6604d5651fa3a9b35b74d9b3c8247cf290ccb944de4afaeb4,116c4d5fd6fe6d897e7008c7497d6220e2ddc3f0c5f41dd34129b82f33ddfc61,2019-04-01,Ted955 Reilly981,Fitchburg Outpatient Clinic,43,3,1,1,1.3590697674418604,23847.94,554.6032558139534,1,0,0.0,14.33
1dd4caf0181c01c6604d5651fa3a9b35b74d9b3c8247cf290ccb944de4afaeb4,116c4d5fd6fe6d897e7008c7497d6220e2ddc3f0c5f41dd34129b82f33ddfc61,2005-10-01,Ted955 Reilly981,Fitchburg Outpatient Clinic,41,2,1,1,2.044085365853659,19359.399999999998,472.180487804878,1,0,0.0,20.5
1dd4caf0181c01c6604d5651fa3a9b35b74d9b3c8247cf290ccb944de4afaeb4,116c4d5fd6fe6d897e7008c7497d6220e2ddc3f0c5f41dd34129b82f33ddfc61,2017-05-01,Ted955 Reilly981,Fitchburg Outpatient Clinic,41,4,2,0,1.737059620596206,68915.45999999999,1680.8648780487804,2,1,50.0,10.25
1dd4caf0181c01c6604d5651fa3a9b35b74d9b3c8247cf290ccb944de4afaeb4,116c4d5fd6fe6d897e7008c7497d6220e2ddc3f0c5f41dd34129b82f33ddfc61,2009-01-01,Ted955 Reilly981,Fitchburg Outpatient Clinic,40,4,1,1,1.1323680555555555,57997.34999999999,1449.9337499999997,1,0,0.0,10.0
1dd4caf0181c01c6604d5651fa3a9b35b74d9b3c8247cf290ccb944de4afaeb4,116c4d5fd6fe6d897e7008c7497d6220e2ddc3f0c5f41dd34129b82f33ddfc61,2017-07-01,Ted955 Reilly981,Fitchburg Outpatient Clinic,37,3,1,2,2.2473948948948945,27908.71,754.2894594594594,1,0,0.0,12.33
1dd4caf0181c01c6604d5651fa3a9b35b74d9b3c8247cf290ccb944de4afaeb4,116c4d5fd6fe6d897e7008c7497d6220e2ddc3f0c5f41dd34129b82f33ddfc61,2008-12-01,Ted955 Reilly981,Fitchburg Outpatient Clinic,33,2,0,0,0.25,4705.139999999999,142.57999999999998,0,0,0.0,16.5
1dd4caf0181c01c6604d5651fa3a9b35b74d9b3c8247cf290ccb944de4afaeb4,116c4d5fd6fe6d897e7008c7497d6220e2ddc3f0c5f41dd34129b82f33ddfc61,2009-02-01,Ted955 Reilly981,Fitchburg Outpatient Clinic,33,3,0,0,0.25757575757575757,6379.089999999999,193.30575757575755,0,0,0.0,11.0


In [0]:
kpi_rows = [
    (
        "Total Patients",
        "Distinct synthetic patients in the analytical population.",
        "Executive"
    ),
    (
        "Total Encounters",
        "Count of healthcare encounters across all encounter classes.",
        "Executive"
    ),
    (
        "30-Day Readmission Rate",
        "Percentage of inpatient index discharges followed by another inpatient admission within 30 days.",
        "Outcomes"
    ),
    (
        "Average Encounter Duration",
        "Average elapsed hours between encounter start and encounter end.",
        "Operations"
    ),
    (
        "Total Claim Cost",
        "Sum of encounter-level total claim cost.",
        "Financial"
    ),
    (
        "Patient Responsibility",
        "Total claim cost less payer coverage, floored at zero.",
        "Financial"
    ),
    (
        "Payer Coverage %",
        "Payer coverage divided by total claim cost.",
        "Financial"
    ),
    (
        "Operational Risk Tier",
        "Rule-based segmentation using readmissions, inpatient utilization, and active condition burden. Not a predictive model.",
        "Patient Outcomes"
    )
]

kpi_dictionary = spark.createDataFrame(
    kpi_rows,
    [
        "kpi_name",
        "business_definition",
        "business_domain"
    ]
)

save_mart(
    kpi_dictionary,
    "mart_kpi_dictionary"
)

Created healthcare_analytics.gold.mart_kpi_dictionary


In [0]:
display(
    spark.sql("""
        SHOW TABLES IN healthcare_analytics.gold
    """)
)

database,tableName,isTemporary
gold,dim_date,false
gold,dim_diagnosis,false
gold,dim_facility,false
gold,dim_patient,false
gold,dim_payer,false
gold,dim_provider,false
gold,fact_claim,false
gold,fact_encounter,false
gold,fact_procedure,false
gold,fact_readmission,false


In [0]:
display(
    spark.sql("""
        SHOW TABLES IN healthcare_analytics.gold
    """)
)

database,tableName,isTemporary
gold,dim_date,false
gold,dim_diagnosis,false
gold,dim_facility,false
gold,dim_patient,false
gold,dim_payer,false
gold,dim_provider,false
gold,fact_claim,false
gold,fact_encounter,false
gold,fact_procedure,false
gold,fact_readmission,false


In [0]:
# ============================================================
# EXECUTIVE HEALTHCARE MART - FINAL
# ============================================================

enc = spark.table(
    "healthcare_analytics.gold.fact_encounter"
)

patients = spark.table(
    "healthcare_analytics.gold.dim_patient"
)

readm = spark.table(
    "healthcare_analytics.gold.fact_readmission"
)


# Encounter KPIs
enc_metrics = enc.agg(

    F.count("*").alias(
        "total_encounters"
    ),

    F.countDistinct(
        "patient_id"
    ).alias(
        "patients_with_encounters"
    ),

    F.sum(
        F.when(
            F.col("is_inpatient") == True,
            1
        ).otherwise(0)
    ).alias(
        "inpatient_encounters"
    ),

    F.avg(
        "encounter_duration_hours"
    ).alias(
        "avg_encounter_duration_hours"
    ),

    F.sum(
        "total_claim_cost"
    ).alias(
        "total_claim_cost"
    ),

    F.sum(
        "payer_coverage"
    ).alias(
        "total_payer_coverage"
    ),

    F.sum(
        "patient_responsibility"
    ).alias(
        "total_patient_responsibility"
    ),

    F.avg(
        "total_claim_cost"
    ).alias(
        "avg_cost_per_encounter"
    )

).first()


# Patient KPIs
patient_metrics = patients.agg(

    F.count("*").alias(
        "total_patients"
    ),

    F.sum(
        F.when(
            F.col("is_deceased") == True,
            1
        ).otherwise(0)
    ).alias(
        "deceased_patients"
    )

).first()


# Readmission KPIs
readmission_metrics = readm.agg(

    F.count("*").alias(
        "eligible_inpatient_discharges"
    ),

    F.sum(
        "readmission_30d_flag"
    ).alias(
        "readmissions_30d"
    )

).first()


eligible = (
    readmission_metrics[
        "eligible_inpatient_discharges"
    ] or 0
)

readmissions = (
    readmission_metrics[
        "readmissions_30d"
    ] or 0
)

readmission_rate_pct = (
    (readmissions / eligible) * 100
    if eligible > 0
    else 0.0
)


executive = spark.createDataFrame(

    [(
        int(patient_metrics["total_patients"] or 0),

        int(enc_metrics["total_encounters"] or 0),

        int(
            enc_metrics[
                "patients_with_encounters"
            ] or 0
        ),

        int(
            enc_metrics[
                "inpatient_encounters"
            ] or 0
        ),

        float(
            enc_metrics[
                "avg_encounter_duration_hours"
            ] or 0
        ),

        float(
            enc_metrics[
                "total_claim_cost"
            ] or 0
        ),

        float(
            enc_metrics[
                "total_payer_coverage"
            ] or 0
        ),

        float(
            enc_metrics[
                "total_patient_responsibility"
            ] or 0
        ),

        float(
            enc_metrics[
                "avg_cost_per_encounter"
            ] or 0
        ),

        int(eligible),

        int(readmissions),

        float(readmission_rate_pct),

        int(
            patient_metrics[
                "deceased_patients"
            ] or 0
        )

    )],

    [
        "total_patients",
        "total_encounters",
        "patients_with_encounters",
        "inpatient_encounters",
        "avg_encounter_duration_hours",
        "total_claim_cost",
        "total_payer_coverage",
        "total_patient_responsibility",
        "avg_cost_per_encounter",
        "eligible_inpatient_discharges",
        "readmissions_30d",
        "readmission_rate_pct",
        "deceased_patients"
    ]
)

executive = (
    executive
    .withColumn(
        "payer_coverage_pct",
        F.round(
            F.when(
                F.col("total_claim_cost") > 0,

                F.col("total_payer_coverage")
                / F.col("total_claim_cost")
                * 100

            ).otherwise(0),
            2
        )
    )
    .withColumn(
        "patient_responsibility_pct",
        F.round(
            F.when(
                F.col("total_claim_cost") > 0,

                F.col("total_patient_responsibility")
                / F.col("total_claim_cost")
                * 100

            ).otherwise(0),
            2
        )
    )
    .withColumn(
        "mart_refreshed_at",
        F.current_timestamp()
    )
)


save_mart(
    executive,
    "mart_executive_healthcare"
)

display(executive)

Created healthcare_analytics.gold.mart_executive_healthcare


total_patients,total_encounters,patients_with_encounters,inpatient_encounters,avg_encounter_duration_hours,total_claim_cost,total_payer_coverage,total_patient_responsibility,avg_cost_per_encounter,eligible_inpatient_discharges,readmissions_30d,readmission_rate_pct,deceased_patients,payer_coverage_pct,patient_responsibility_pct,mart_refreshed_at
1138,63795,1138,993,6.025351377241343,1.7301086649000028E8,1.2499112377999987E8,4.801974270999998E7,2711.981604984721,993,188,18.932527693856997,138,72.24,27.76,2026-09-07T20:54:09.438Z


In [0]:
display(
    spark.sql("""
        SHOW TABLES
        IN healthcare_analytics.gold
    """)
)

database,tableName,isTemporary
gold,dim_date,false
gold,dim_diagnosis,false
gold,dim_facility,false
gold,dim_patient,false
gold,dim_payer,false
gold,dim_provider,false
gold,fact_claim,false
gold,fact_encounter,false
gold,fact_procedure,false
gold,fact_readmission,false


In [0]:
marts = spark.sql("""
SELECT table_name
FROM healthcare_analytics.information_schema.tables
WHERE table_schema = 'gold'
  AND table_name LIKE 'mart_%'
ORDER BY table_name
""")

display(marts)

print("Analytics marts:", marts.count())

table_name
mart_claims_cost
mart_clinical_utilization
mart_executive_healthcare
mart_hospital_operations
mart_kpi_dictionary
mart_patient_outcomes
mart_provider_performance
mart_readmissions


Analytics marts: 8
